# Word Embeddings: Ungraded Practice Notebook

In this ungraded notebook, you'll try out all the individual techniques that you learned about in the lecture. Practicing on small examples will prepare you for the graded assignment, where you will combine the techniques in more advanced ways to create word embeddings from a real-life corpus.

This notebook is made of two main parts: data preparation, and the continuous bag-of-words (CBOW) model.

To get started, import and initialize all the libraries you will need.

In [3]:
import re
import nltk
from nltk.tokenize import word_tokenize
import emoji
import numpy as np

from utils import get_dict
import utils

nltk.download('punkt')  # download pre-trained Punkt tokenizer for English

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mrinm\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

# Data preparation
In the data preparation phase, starting with a corpus of text, you will:

- Clean and tokenize the corpus.

- Extract the pairs of context words and center word that will make up the training data set for the CBOW model. The context words are the features that will be fed into the model, and the center words are the target values that the model will learn to predict.

- Create simple vector representations of the context words (features) and center words (targets) that can be used by the neural network of the CBOW model.

## Cleaning and tokenization

To demonstrate the cleaning and tokenization process, consider a corpus that contains emojis and various punctuation signs.

In [8]:
corpus = 'Who ❤️ "word embeddings" in 2020? I do!!!'

Tokenize the corpus using utility function that you created earlier

In [9]:
words = utils.tokenize(corpus)
words

['who', '❤️', 'word', 'embeddings', 'in', '.', 'i', 'do', '.']

## Sliding window of words

getting center and context words from the single corpous with half context size = 2

In [10]:
for x, y in utils.get_windows(
            words,
            2
        ):
    print(f'{x}\t{y}')

['who', '❤️', 'embeddings', 'in']	word
['❤️', 'word', 'in', '.']	embeddings
['word', 'embeddings', '.', 'i']	in
['embeddings', 'in', 'i', 'do']	.
['in', '.', 'do', '.']	i


## Transforming words into vectors for the training set

To finish preparing the training set, you need to transform the context words and center words into vectors.

### Mapping words to indices and indices to words

The center words will be represented as one-hot vectors, and the vectors that represent context words are also based on one-hot vectors.

To create one-hot word vectors, you can start by mapping each unique word to a unique integer (or index). We have provided a helper function, `get_dict`, that creates a Python dictionary that maps words to integers and back.

In [11]:
word2Ind, Ind2word = get_dict(words)

### Creating a training set for the example corpus

In [12]:
words = utils.tokenize(corpus)
C = 2
word2Ind, Ind2word = get_dict(words)
V = len(word2Ind) 

for context_words_vector, center_word_vector in utils.get_training_example(words, 2, word2Ind, V):
    print(f'Context words vector:  {context_words_vector}')
    print(f'Center word vector:  {center_word_vector}')
    print()

Context words vector:  [0.   0.   0.25 0.   0.25 0.25 0.   0.25]
Center word vector:  [0. 0. 0. 0. 0. 0. 1. 0.]

Context words vector:  [0.25 0.   0.   0.   0.25 0.   0.25 0.25]
Center word vector:  [0. 0. 1. 0. 0. 0. 0. 0.]

Context words vector:  [0.25 0.   0.25 0.25 0.   0.   0.25 0.  ]
Center word vector:  [0. 0. 0. 0. 1. 0. 0. 0.]

Context words vector:  [0.   0.25 0.25 0.25 0.25 0.   0.   0.  ]
Center word vector:  [1. 0. 0. 0. 0. 0. 0. 0.]

Context words vector:  [0.5  0.25 0.   0.   0.25 0.   0.   0.  ]
Center word vector:  [0. 0. 0. 1. 0. 0. 0. 0.]



# The continuous bag-of-words model

The CBOW model is based on a neural network, the architecture of which looks like the figure below, as you'll recall from the lecture.

<div style="width:image width px; font-size:100%; text-align:center;"><img src='./images/cbow_model_architecture.png?1' alt="alternate text" width="width" height="height" style="width:917;height:337;" /> Figure 1 </div>

This part of the notebook will walk you through:

- The two activation functions used in the neural network.

- Forward propagation.

- Cross-entropy loss.

- Backpropagation.

- Gradient descent.

- Extracting the word embedding vectors from the weight matrices once the neural network has been trained.

## Forward propagation

Let's dive into the neural network itself, which is shown below with all the dimensions and formulas you'll need.

<div style="width:image width px; font-size:100%; text-align:center;"><img src='./images/cbow_model_dimensions_single_input.png?2' alt="alternate text" width="width" height="height" style="width:839;height:349;" /> Figure 2 </div>

Set $N$ equal to 3. Remember that $N$ is a hyperparameter of the CBOW model that represents the size of the word embedding vectors, as well as the size of the hidden layer.

In [13]:
N = 3

Before you start training the neural network, you need to initialize the weight matrices and bias vectors with random values.

In the assignment you will implement a function to do this yourself using `numpy.random.rand`. In this notebook, we've pre-populated these matrices and vectors for you.

### Initialization of the weights and biases

In [14]:
W1 = np.array([[ 0.41687358,  0.08854191, -0.23495225,  0.28320538,  0.41800106],
               [ 0.32735501,  0.22795148, -0.23951958,  0.4117634 , -0.23924344],
               [ 0.26637602, -0.23846886, -0.37770863, -0.11399446,  0.34008124]])

W2 = np.array([[-0.22182064, -0.43008631,  0.13310965],
               [ 0.08476603,  0.08123194,  0.1772054 ],
               [ 0.1871551 , -0.06107263, -0.1790735 ],
               [ 0.07055222, -0.02015138,  0.36107434],
               [ 0.33480474, -0.39423389, -0.43959196]])

b1 = np.array([[ 0.09688219],
               [ 0.29239497],
               [-0.27364426]])

b2 = np.array([[ 0.0352008 ],
               [-0.36393384],
               [-0.12775555],
               [-0.34802326],
               [-0.07017815]])

In [15]:
print(f'V (vocabulary size): {V}')
print(f'N (embedding size / size of the hidden layer): {N}')
print(f'size of W1: {W1.shape} (NxV)')
print(f'size of b1: {b1.shape} (Nx1)')
print(f'size of W2: {W2.shape} (VxN)')
print(f'size of b2: {b2.shape} (Vx1)')

V (vocabulary size): 8
N (embedding size / size of the hidden layer): 3
size of W1: (3, 5) (NxV)
size of b1: (3, 1) (Nx1)
size of W2: (5, 3) (VxN)
size of b2: (5, 1) (Vx1)
